In [2]:
import os
import logging
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import tempfile

TMPDIR = tempfile.mkdtemp()

src_path = Path.cwd() / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

# --- logging setup ---

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# optionally raise the level for noisy third-party loggers
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("great_expectations._docs_decorators").setLevel(logging.WARNING)

logger = logging.getLogger("notebook")
logger.info("logging is configured")

2026-10-08 08:37:55 INFO [notebook] logging is configured


Added to sys.path: /home/jtv/code/jtviegas/meddra/src


In [ ]:
dataset_name: str = "jtviegas/faers_bronze_drug"

In [ ]:

ds: DataFrameSplits = store.get(dataset_name)
df = ds.train[-1024:]

__________________________________________________________________

In [ ]:
from typing import Any
from tgedr_dataops_abs.store import NoStoreException
from tgedr_dataops.store.contracted_store import (
    HuggingFaceDatasetFileBasedStore, ContractedHFDatasetFileBasedStore,
    DataFrameSplits,
  )


class IsProcessingDue:

  def __init__(self) -> None:
    #super().__init__(configuration=configuration)
    self._store: HuggingFaceDatasetFileBasedStore = HuggingFaceDatasetFileBasedStore()

  
  def _find_preferred_terms_actual_time(self, table_preferred_terms) -> int:
    logger.info("[_find_preferred_terms_actual_time|in]")
    result: int = 0
    # read the dataset
    try:
        result = self._store.get(table=table_preferred_terms).select("actual_time").take(1)[0].actual_time
    except NoStoreException as nse:
        # no dataset there
        logger.warning(f"[_find_preferred_terms_actual_time] => {nse}")

    logger.info(f"[_find_preferred_terms_actual_time|out] => {result}")
    return result
  
  def extract(self, table_preferred_terms: str, input_files_location: str, smq_mapping_files_location: str) -> Any:

    logger.info(f"[extract|in] (table_preferred_terms={table_preferred_terms}, input_files_location={input_files_location}, "
                f"smq_mapping_files_location={smq_mapping_files_location})")

    # find latest preferred terms processed release version
    pt_actual_time: int = self._find_preferred_terms_actual_time(table_preferred_terms)

    # # find latest meddra files modification time
    # meddra_files_modification_time: int = self._find_meddra_files_modification_ts()

    # # find latest SMQ-NNMQ mapping file modification time
    # smq_nn_mapping_files_modification_time: int = self._find_smq_nn_files_modification_ts()

    # # if any of them (latest meddra files or latest SMQ-NNMQ mapping file)
    # # differ from the preferred terms dataset `actual_time` then we need to reprocess
    # decision: bool = (pt_actual_time < meddra_files_modification_time) or (
    #     pt_actual_time < smq_nn_mapping_files_modification_time
    # )
    # self._output["processing_due"] = "1" if decision == True else "0"
    # self._output["updated_ts"] = str(max(meddra_files_modification_time, smq_nn_mapping_files_modification_time))

    logger.info("[extract|out]")